In [1]:
#QB ML MODEL
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import MinMaxScaler

pd.options.mode.chained_assignment = None

import sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

from sklearn.model_selection import train_test_split
import joblib


#scaler to scale data
scaler = MinMaxScaler()

#read csv files into pandas
dfFantasy = pd.read_pickle("PickleFiles/final_qb_data.pkl")
dfFantasy.replace([np.inf, -np.inf], np.nan, inplace=True)
numeric_cols = dfFantasy.select_dtypes(include=[np.number]).columns
for column in numeric_cols:
    dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)
dfGrades = pd.read_pickle("PickleFiles/AVbyPositionGroup.pkl")

def correctData(df, pprTF):
  #cols to make per game
  cols = ['completions', 'attempts', 'passing_yards',
       'passing_tds', 'interceptions', 'sacks',
       'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch',
       'passing_first_downs', 'passing_2pt_conversions',
       'carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs',
       'rushing_2pt_conversions', 'fantasy_points', 'age']

  #basing data if ppr or not
  if pprTF == 2:
    pass
  elif pprTF == 0:
    pass
  elif pprTF == 1:
    pass

    
  #adding ppg column
  df.loc[:, 'PPG'] = df['fantasy_points'] / df['GP']


  #make all columns in a per game basis
  for col in cols:
    df.loc[:, col] = df[col] / df['GP'] 


  #only players with more than 7 games.
  df = df[df.GP > 7]
  df = df[df.fantasy_points >= 0]

  df = df[df.PPG > 5]
  

  return df

#removes unneccesary stats
def removeUnwanted(dfPos, pos):
  dfPos = dfPos.drop(columns=['season',"GP", "season_type", "fantasy_points", "player_display_name", "player_id", "team", "position"])
  return dfPos

#shifts data forward one year
def makeCorrectShift(df):
  shifters = ['PPG','season','GP','season_type','age','fantasy_points','completions','attempts','passing_yards','passing_tds','interceptions','sacks','sack_fumbles_lost','passing_air_yards','passing_yards_after_catch','passing_first_downs','passing_2pt_conversions','carries','rushing_yards','rushing_tds','rushing_fumbles_lost','rushing_first_downs','rushing_2pt_conversions']
  
  #adds target variable
  df["PPG"] = df["PPG"]
  
  #shifts it forward a year (for example 2011 goes to 2012)
  df[shifters] = df.groupby('player_display_name')[shifters].shift(1)
  df = df.dropna()

  return df

#where machine learning is done. returns the model and score.
from sklearn.inspection import permutation_importance

def machineLearning(df, arr, dictParam):
    # Define predictors excluding the target variable
    predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']

    # Split the data
    x = df[predictors].values
    y = df["PPG"].values
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

    # Initialize and train GradientBoostingRegressor
    gbr = GradientBoostingRegressor(**dictParam)
    gbr.fit(x_train, y_train)

    # Evaluate the model
    predict_test = gbr.predict(x_test)
    mae = mean_absolute_error(y_test, predict_test)

    predict_test_unscaled = predict_test * (arr[1] - arr[0]) + arr[0]
    y_test_unscaled = y_test * (arr[1] - arr[0]) + arr[0]

    # Calculate permutation importance
    r = permutation_importance(gbr, x_test, y_test, n_repeats=100, random_state=0)

    # Organize importances
    importance_dict = {name: score for name, score in zip(predictors, r.importances_mean)}
    sorted_importances = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
    
    for feature, importance in sorted_importances:
      print(f"{feature}: {importance}")


    return [mae, gbr]

# Example usage of the modified function


def getBestParams(df, arr):

  #make the predictors and data and test sets correctly
  predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']
  x = df[predictors].values
  y = df["PPG"].values
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=40)

  
  #make the parameters to search over. for hidden_layer_sizes, I experimented with alot and the ones listed now is just final set of experiment.
  
  grid = {
      'n_estimators': [100, 200, 300],
      'learning_rate': [0.01, 0.1, 0.2],
      'max_depth': [3, 4, 5],
      'min_samples_split': [2, 3, 4]
  }

  #create an MLPRegressor object
  gbr = GradientBoostingRegressor()

  #create a GridSearchCV object and fit it to the training data
  grid_search = GridSearchCV(gbr, param_grid=grid, cv=5, n_jobs=-1)
  grid_search.fit(x_train, y_train)

  #the best model to make predictions on the test data and evaluate performance
  y_pred = grid_search.predict(x_test)

  #inverse transform the scaled predictions to get the original scale, uses a reverse of original formula
  for i in range(len(y_pred)):
    y_pred[i] = (y_pred[i]*(arr[1] - arr[0])) + arr[0]
  for i in range(len(y_test)):
    y_test[i] = (y_test[i]*(arr[1] - arr[0])) + arr[0]


  print(mean_absolute_error(y_test, y_pred))

  return grid_search.best_params_

#gets original value for fantasy points for predictions.
def getScaleBack(df):
  #index of column
  column_index = df.columns.get_loc("PPG")

  #min value of column:
  min_value = df["PPG"].min()

  #scaling valye of column
  #scaling_factor = scaler.scale_[column_index]
  max_value = df["PPG"].max()

  #array to be used later to scale each data
  arr = [min_value, max_value]

  return arr

def test(df, model, arr):
  #make columns everything but target
  predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']


  #make train and test sets
  x = df[predictors].values
  y = df["PPG"].values
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=40)

  #make the predictions
  predict_test = model.predict(x_test)

  #inverse transform the scaled predictions to get the original scale by reversing formula
  for i in range(len(predict_test)):
    predict_test[i] = (predict_test[i]*(arr[1] - arr[0])) + arr[0]
  for i in range(len(y_test)):
    y_test[i] = (y_test[i]*(arr[1] - arr[0])) + arr[0]

  #average error 
  mae = mean_absolute_error(y_test, predict_test)
  print("test ", mae)

#if ppr is 0, than it is non ppr. if 1, then it is half ppr. if 2, full ppr. loops through each.
for ppr in [0,1,2]:

  dfFantasyCopy = dfFantasy.copy()

  dfFantasyCopy = correctData(dfFantasyCopy, ppr)

  dfFantasyCopy = makeCorrectShift(dfFantasyCopy)

  dfFantasyCopy = dfFantasyCopy.loc[dfFantasyCopy["season"] != 2012]

  dfFantasyCopy = removeUnwanted(dfFantasyCopy, "QB")

  dfFantasyCopy = dfFantasyCopy.reset_index(drop=True)

  #gets fantasy_points_ppr scale per each position
  scaleQB = getScaleBack(dfFantasyCopy)

  dfFantasyCopy[dfFantasyCopy.columns] = scaler.fit_transform(dfFantasyCopy[dfFantasyCopy.columns])

  #obtained by running the getBestParams function per each respective position
  paramQB = getBestParams(dfFantasyCopy, scaleQB)

  #makes array of model and score, then prints it
  qbArray = machineLearning(dfFantasyCopy, scaleQB, paramQB)
  num = qbArray[0]
  qbModel = qbArray[1]
  print("qb score(ppg off on average per player): ", num)

  if ppr == 0:
      joblib.dump(qbModel, "qb models/qbModelNonPPR.joblib")
  elif ppr == 1:
      joblib.dump(qbModel, "qb models/qbModelHalfPPR.joblib")
  elif ppr == 2:
      joblib.dump(qbModel, "qb models/qbModelPPR.joblib")



/Users/kmaran3/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/462228906.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/462228906.py:54: FutureWarning: Setting an item of

0.6897639181006948
passing_tds: 0.918896917109077
passing_yards: 0.137750342741192
rushing_yards: 0.04551422314386338
rushing_tds: 0.039851980585076635
passing_first_downs: 0.028122251059792144
sacks: 0.026260627999539862
interceptions: 0.024260637666085508
carries: 0.016072530973192722
passing_yards_after_catch: 0.007120174115884859
rushing_first_downs: 0.005904724612114271
rushing_fumbles_lost: 0.005117060160604312
passing_2pt_conversions: 0.002296390398382291
sack_fumbles_lost: 0.001422146680503006
completions: 0.0009048026861979353
rb: 0.0006768983490623503
rushing_2pt_conversions: 0.0004752305628524456
qb: 1.845160287396008e-05
wrte: -4.3508106088173684e-05
attempts: -0.00030833768421487107
oline: -0.0007027216601927322
passing_air_yards: -0.0012554420608615912
age: -0.006968034794958106
dst: -0.015695606603403234
qb score(ppg off on average per player):  0.04042213230167958


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/462228906.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.33333333  7.5        19.5        28.125      24.6875     19.8
 23.75       23.3125     25.125      24.25       24.0625     23.4375
 23.3125     25.0625     28.52941176 28.82352941 11.          7.11111111
  8.         27.875      28.5        28.53333333 29.4375     24.125
 24.26666667 25.54545455 22.91666667  0.         18.125       0.66666667
  6.33333333 18.625      16.72727273 23.25       18.         20.53846154
 15.          1.5        22.625      23.5        21.375      24.26666667
 23.42857143 22.8        20.26666667 20.75        3.          8.
 21.9         2.5        26.          0.33333333  2.5        10.
  0.         19.8125     23.6875     24.1875     23.5625     23.46666667
 23.75       22.75        0.         16.         23.4375     25.5
 26.58333333 23.42857143 24.06666667

0.7201541409344794
passing_tds: 0.8995814932881564
passing_yards: 0.14597913105827945
rushing_tds: 0.055749961278339415
rushing_yards: 0.041373605584007614
passing_first_downs: 0.02170208142110646
passing_yards_after_catch: 0.020356760591214114
interceptions: 0.019428830453224334
sacks: 0.017461964541948566
rushing_first_downs: 0.015984034834042638
carries: 0.010583633132003431
oline: 0.003002864465722254
sack_fumbles_lost: 0.0027591131919697874
completions: 0.0024563267066167837
passing_2pt_conversions: 0.001566400337583793
attempts: 0.0014759903826530984
passing_air_yards: 0.0009550766482624173
rushing_2pt_conversions: 0.0008152715428225265
age: 0.0007489633109742954
rb: 0.0007333011740168883
qb: 0.00028778175284152277
wrte: 0.00020705490699608807
rushing_fumbles_lost: -0.00012157763556693402
dst: -0.006829414462545873
qb score(ppg off on average per player):  0.040411849074104514


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/462228906.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 2.33333333  7.5        19.5        28.125      24.6875     19.8
 23.75       23.3125     25.125      24.25       24.0625     23.4375
 23.3125     25.0625     28.52941176 28.82352941 11.          7.11111111
  8.         27.875      28.5        28.53333333 29.4375     24.125
 24.26666667 25.54545455 22.91666667  0.         18.125       0.66666667
  6.33333333 18.625      16.72727273 23.25       18.         20.53846154
 15.          1.5        22.625      23.5        21.375      24.26666667
 23.42857143 22.8        20.26666667 20.75        3.          8.
 21.9         2.5        26.          0.33333333  2.5        10.
  0.         19.8125     23.6875     24.1875     23.5625     23.46666667
 23.75       22.75        0.         16.         23.4375     25.5
 26.58333333 23.42857143 24.06666667

0.7323581514414255
passing_tds: 0.8891949273566913
passing_yards: 0.15056006594564494
rushing_tds: 0.05884900352620979
rushing_yards: 0.04564737822224139
interceptions: 0.025130994642237607
passing_yards_after_catch: 0.02210547161812111
passing_first_downs: 0.021762985741846114
sacks: 0.019840710979675377
rushing_first_downs: 0.017146420060928935
carries: 0.005519222835447502
sack_fumbles_lost: 0.003402280217252576
oline: 0.0027715393947504783
passing_2pt_conversions: 0.0023687330427483665
completions: 0.0021576369835549438
rushing_2pt_conversions: 0.0007405135131758034
age: 0.0003247817828465982
qb: -0.0001302259547787099
wrte: -0.00039341344834423306
attempts: -0.000460861355937936
passing_air_yards: -0.0006838676118615917
rushing_fumbles_lost: -0.0010280943061982495
rb: -0.0053222891571111704
dst: -0.00593040702634137
qb score(ppg off on average per player):  0.040006429091587664


In [2]:
#RB ML MODEL
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import MinMaxScaler

pd.options.mode.chained_assignment = None

import sklearn
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

from sklearn.model_selection import train_test_split
import joblib


#scaler to scale data
scaler = MinMaxScaler()

#read csv files into pandas
dfFantasy = pd.read_pickle("PickleFiles/final_rb_data.pkl")
dfFantasy.replace([np.inf, -np.inf], np.nan, inplace=True)
numeric_cols = dfFantasy.select_dtypes(include=[np.number]).columns
for column in numeric_cols:
    dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)
dfGrades = pd.read_pickle("PickleFiles/AVbyPositionGroup.pkl")

def correctData(df, pprTF):
  #cols to make per game
  cols = ['carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_2pt_conversions', 'receptions', 'targets',
       'receiving_yards', 'receiving_tds',
       'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs',
       'receiving_2pt_conversions', 'special_teams_tds', 'fantasy_points', 'rrtd', 'age']

  #basing data if ppr or not
  if pprTF == 2:
    pass
  elif pprTF == 0:
    df.loc[:, "fantasy_points"] = df["fantasy_points"] - df["receptions"]
  elif pprTF == 1:
    df.loc[:, "fantasy_points"] = df["fantasy_points"] - (df["receptions"]/2)

    
  #adding ppg column
  df.loc[:, 'PPG'] = df['fantasy_points'] / df['GP']


  #make all columns in a per game basis
  for col in cols:
    df.loc[:, col] = df[col] / df['GP'] 


  #only players with more than 7 games.
  df = df[df.GP > 7]
  df = df[df.fantasy_points >= 0]

  df = df[df.PPG > 2]
  

  return df

#removes unneccesary stats
def removeUnwanted(dfPos, pos):
  dfPos = dfPos.drop(columns=['season',"GP", "season_type", "fantasy_points", "player_display_name", "player_id", "team", "position"])
  return dfPos

#shifts data forward one year
def makeCorrectShift(df):
  shifters = ['player_id', 'season', 'player_display_name', 'team', 'GP', 'position',
       'age','season_type', 'carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs',
       'rushing_2pt_conversions', 'receptions', 'targets',
       'receiving_yards', 'receiving_tds',
       'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs',
       'receiving_2pt_conversions', 'special_teams_tds', 'fantasy_points',
       'rrtd']
  
  #adds target variable
  df["PPG"] = df["PPG"]
  
  #shifts it forward a year (for example 2011 goes to 2012)
  df[shifters] = df.groupby('player_display_name')[shifters].shift(1)
  df = df.dropna()

  return df

#where machine learning is done. returns the model and score.
from sklearn.inspection import permutation_importance

def machineLearning(df, arr, dictParam):
    # Define predictors excluding the target variable
    predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']

    # Split the data
    x = df[predictors].values
    y = df["PPG"].values
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

    # Initialize and train GradientBoostingRegressor
    gbr = GradientBoostingRegressor(**dictParam)
    gbr.fit(x_train, y_train)

    # Evaluate the model
    predict_test = gbr.predict(x_test)
    mae = mean_absolute_error(y_test, predict_test)

    predict_test_unscaled = predict_test * (arr[1] - arr[0]) + arr[0]
    y_test_unscaled = y_test * (arr[1] - arr[0]) + arr[0]

    # print("Predicted vs Actual PPG (unscaled):")
    # for pred, actual in zip(predict_test_unscaled, y_test_unscaled):
    #     print(f"Predicted: {pred:.2f}, Actual: {actual:.2f}")

    # Calculate permutation importance
    r = permutation_importance(gbr, x_test, y_test, n_repeats=100, random_state=0)

    # Organize importances
    importance_dict = {name: score for name, score in zip(predictors, r.importances_mean)}
    sorted_importances = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

    for feature, importance in sorted_importances:
        print(f"{feature}: {importance}")
    


    return [mae, gbr, sorted_importances]

# Example usage of the modified function


def getBestParams(df, arr):

  #make the predictors and data and test sets correctly
  predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']
  x = df[predictors].values
  y = df["PPG"].values
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=40)

  
  #make the parameters to search over. for hidden_layer_sizes, I experimented with alot and the ones listed now is just final set of experiment.
  
  grid = {
      'n_estimators': [100, 200, 300],
      'learning_rate': [0.01, 0.1, 0.2],
      'max_depth': [3, 4, 5],
      'min_samples_split': [2, 3, 4]
  }

  #create an MLPRegressor object
  gbr = GradientBoostingRegressor()

  #create a GridSearchCV object and fit it to the training data
  grid_search = GridSearchCV(gbr, param_grid=grid, cv=5, n_jobs=-1)
  grid_search.fit(x_train, y_train)

  #the best model to make predictions on the test data and evaluate performance
  y_pred = grid_search.predict(x_test)

  #inverse transform the scaled predictions to get the original scale, uses a reverse of original formula
  for i in range(len(y_pred)):
    y_pred[i] = (y_pred[i]*(arr[1] - arr[0])) + arr[0]
  for i in range(len(y_test)):
    y_test[i] = (y_test[i]*(arr[1] - arr[0])) + arr[0]


  # print(mean_absolute_error(y_test, y_pred))

  return grid_search.best_params_

#gets original value for fantasy points for predictions.
def getScaleBack(df):
  #index of column
  column_index = df.columns.get_loc("PPG")

  #min value of column:
  min_value = df["PPG"].min()

  #scaling valye of column
  #scaling_factor = scaler.scale_[column_index]
  max_value = df["PPG"].max()

  #array to be used later to scale each data
  arr = [min_value, max_value]

  return arr

def test(df, model, arr):
  #make columns everything but target
  predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']


  #make train and test sets
  x = df[predictors].values
  y = df["PPG"].values
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=40)

  #make the predictions
  predict_test = model.predict(x_test)

  #inverse transform the scaled predictions to get the original scale by reversing formula
  for i in range(len(predict_test)):
    predict_test[i] = (predict_test[i]*(arr[1] - arr[0])) + arr[0]
  for i in range(len(y_test)):
    y_test[i] = (y_test[i]*(arr[1] - arr[0])) + arr[0]

  #average error 
  mae = mean_absolute_error(y_test, predict_test)
  print("test ", mae)

#if ppr is 0, than it is non ppr. if 1, then it is half ppr. if 2, full ppr. loops through each.
for ppr in [0,1,2]:

  dfFantasyCopy = dfFantasy.copy()

  dfFantasyCopy = correctData(dfFantasyCopy, ppr)

  dfFantasyCopy = makeCorrectShift(dfFantasyCopy)

  dfFantasyCopy = dfFantasyCopy.loc[dfFantasyCopy["season"] != 2012]

  dfFantasyCopy = removeUnwanted(dfFantasyCopy, "RB")

  dfFantasyCopy = dfFantasyCopy.reset_index(drop=True)

  #gets fantasy_points_ppr scale per each position
  scaleRB = getScaleBack(dfFantasyCopy)

  dfFantasyCopy[dfFantasyCopy.columns] = scaler.fit_transform(dfFantasyCopy[dfFantasyCopy.columns])

  #obtained by running the getBestParams function per each respective position
  paramRB = getBestParams(dfFantasyCopy, scaleRB)

  #makes array of model and score, then prints it
  rbArray = machineLearning(dfFantasyCopy, scaleRB, paramRB)
  num = rbArray[0]
  rbModel = rbArray[1]

  print("rb score(ppg off on average per player): ", num)
  if ppr == 0:
      joblib.dump(rbModel, "rb models/rbModelNonPPR.joblib")
  elif ppr == 1:
      joblib.dump(rbModel, "rb models/rbModelHalfPPR.joblib")
  elif ppr == 2:
      joblib.dump(rbModel, "rb models/rbModelPPR.joblib")
#print(dfFantasyRB.columns)

/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[12.54545455  0.84615385 13.08333333 ...  2.25       12.13333333
  6.75      ]' has dtype incompatible with int32, please explicitly cast

rushing_first_downs: 0.11240070811152131
rb: 0.05935220454105317
rushing_yards: 0.04310340376363271
carries: 0.007321576842999732
receiving_yards_after_catch: 0.0056806538374514935
rushing_fumbles_lost: 0.001999526296236719
oline: 0.0017851430128682521
receiving_yards: 0.001626208144449388
receiving_2pt_conversions: 0.0010147725110816596
qb: 0.00047390929067007704
wrte: 0.00028720074281102345
special_teams_tds: 0.00014505397444333545
targets: 0.0001360816200175985
receiving_fumbles_lost: 0.0
rushing_2pt_conversions: -8.34055284697488e-06
receiving_air_yards: -0.0005002649587003394
receiving_tds: -0.001193448939206735
rushing_tds: -0.0017109507246059418
dst: -0.003911078951511802
receptions: -0.003927521173766336
receiving_first_downs: -0.004945353872256293
age: -0.006207670397095692
rrtd: -0.00627964831459618
rb score(ppg off on average per player):  0.13570436126195104


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[12.54545455  0.84615385 13.08333333 ...  2.25       12.13333333
  6.75      ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.18181818 0.         0.5        ... 0.         0.66666667 0.0625    ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. V

rushing_yards: 0.19502074010158718
rushing_first_downs: 0.09193526478395155
rb: 0.052238283865874084
age: 0.025259568279548096
receiving_yards_after_catch: 0.021487487722963165
carries: 0.02050568080731381
rrtd: 0.01873272462458373
receiving_first_downs: 0.014050078063404237
rushing_fumbles_lost: 0.006689650040539532
oline: 0.004322888501916426
dst: 0.001457533059472822
rushing_tds: 0.0007620534531255041
special_teams_tds: 0.0006572264616016788
wrte: 0.00038060611215536835
rushing_2pt_conversions: 0.0002811158981255912
receiving_fumbles_lost: 0.00015618951795991708
receiving_2pt_conversions: 2.489370870428775e-07
qb: -0.00011540530981332008
receiving_tds: -0.0005406770066050259
receiving_yards: -0.0008105954699721674
receptions: -0.0011782392015684373
receiving_air_yards: -0.0017468565920587997
targets: -0.0032726451222288024
rb score(ppg off on average per player):  0.1234744227709289


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[12.54545455  0.84615385 13.08333333 ...  2.25       12.13333333
  6.75      ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.18181818 0.         0.5        ... 0.         0.66666667 0.0625    ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/2857796669.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. V

rushing_yards: 0.17101940838229726
rushing_first_downs: 0.1227339913128565
receiving_first_downs: 0.03384818419883957
rb: 0.0207193554704849
receiving_yards_after_catch: 0.013534195782697756
age: 0.012842135934815762
oline: 0.011311259435142409
rrtd: 0.00792620712397183
wrte: 0.003273828399383897
rushing_fumbles_lost: 0.002926191602202407
targets: 0.0015753101240938983
carries: 0.0013579880000055722
qb: 0.0013482020897491753
rushing_tds: 0.001099816989481327
dst: 0.0009733948199304476
receiving_yards: 0.0008854318163163566
receptions: 0.0006924347932850384
receiving_air_yards: 0.000565047927088197
receiving_tds: 0.00018888703317268885
receiving_2pt_conversions: 0.0
special_teams_tds: 0.0
receiving_fumbles_lost: -0.00017303841636681305
rushing_2pt_conversions: -0.00035699519114615775
rb score(ppg off on average per player):  0.1297822975663568


In [3]:
#WRTE ML MODEL
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import MinMaxScaler

pd.options.mode.chained_assignment = None

import sklearn
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error

from sklearn.model_selection import train_test_split
import joblib


#scaler to scale data
scaler = MinMaxScaler()

#read csv files into pandas
dfFantasy = pd.read_pickle("PickleFiles/final_wrte_data.pkl")
dfFantasy.replace([np.inf, -np.inf], np.nan, inplace=True)
numeric_cols = dfFantasy.select_dtypes(include=[np.number]).columns
for column in numeric_cols:
    dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)
dfGrades = pd.read_pickle("PickleFiles/AVbyPositionGroup.pkl")

def correctData(df, pprTF):
  #cols to make per game
  cols = ['carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_2pt_conversions', 'receptions', 'targets',
       'receiving_yards', 'receiving_tds',
       'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs',
       'receiving_2pt_conversions', 'special_teams_tds', 'fantasy_points', 'rrtd', 'age']

  #basing data if ppr or not
  if pprTF == 2:
    pass
  elif pprTF == 0:
    df.loc[:, "fantasy_points"] = df["fantasy_points"] - df["receptions"]
  elif pprTF == 1:
    df.loc[:, "fantasy_points"] = df["fantasy_points"] - (df["receptions"]/2)

    
  #adding ppg column
  df.loc[:, 'PPG'] = df['fantasy_points'] / df['GP']


  #make all columns in a per game basis
  for col in cols:
    df.loc[:, col] = df[col] / df['GP'] 


  #only players with more than 7 games.
  df = df[df.GP > 7]
  df = df[df.fantasy_points >= 0]

  df = df[df.PPG > 2]
  

  return df

#removes unneccesary stats
def removeUnwanted(dfPos, pos):
  dfPos = dfPos.drop(columns=['season',"GP", "season_type", "fantasy_points", "player_display_name", "player_id", "team", "position"])
  return dfPos

#shifts data forward one year
def makeCorrectShift(df):
  shifters = ['player_id', 'season', 'player_display_name', 'team', 'GP', 'position',
       'age', 'season_type', 'carries', 'rushing_yards', 'rushing_tds',
       'rushing_fumbles_lost', 'rushing_first_downs',
       'rushing_2pt_conversions', 'receptions', 'targets',
       'receiving_yards', 'receiving_tds',
       'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs',
       'receiving_2pt_conversions', 'special_teams_tds', 'fantasy_points',
       'rrtd']
  
  #adds target variable
  df["PPG"] = df["PPG"]
  
  #shifts it forward a year (for example 2011 goes to 2012)
  df[shifters] = df.groupby('player_display_name')[shifters].shift(1)
  df = df.dropna()

  return df

#where machine learning is done. returns the model and score.
from sklearn.inspection import permutation_importance

def machineLearning(df, arr, dictParam):
    # Define predictors excluding the target variable
    predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']

    # Split the data
    x = df[predictors].values
    y = df["PPG"].values
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

    # Initialize and train MLPRegressor
    gbr = GradientBoostingRegressor(**dictParam)
    gbr.fit(x_train, y_train)

    # Evaluate the model
    predict_test = gbr.predict(x_test)
    mae = mean_absolute_error(y_test, predict_test)

    predict_test_unscaled = predict_test * (arr[1] - arr[0]) + arr[0]
    y_test_unscaled = y_test * (arr[1] - arr[0]) + arr[0]

    # print("Predicted vs Actual PPG (unscaled):")
    # for pred, actual in zip(predict_test_unscaled, y_test_unscaled):
    #     print(f"Predicted: {pred:.2f}, Actual: {actual:.2f}")

    # Calculate permutation importance
    r = permutation_importance(gbr, x_test, y_test, n_repeats=100, random_state=0)

    # Organize importances
    importance_dict = {name: score for name, score in zip(predictors, r.importances_mean)}
    sorted_importances = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

    # Print sorted importances
    for feature, importance in sorted_importances:
        print(f"{feature}: {importance}")
    


    return [mae, gbr, sorted_importances]

# Example usage of the modified function


def getBestParams(df, arr):

  #make the predictors and data and test sets correctly
  predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']
  x = df[predictors].values
  y = df["PPG"].values
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=40)

  
  #make the parameters to search over. for hidden_layer_sizes, I experimented with alot and the ones listed now is just final set of experiment.
  
  grid = {
      'n_estimators': [100, 200, 300],
      'learning_rate': [0.01, 0.1, 0.2],
      'max_depth': [3, 4, 5],
      'min_samples_split': [2, 3, 4]
  }

  #create an MLPRegressor object
  gbr = GradientBoostingRegressor()

  #create a GridSearchCV object and fit it to the training data
  grid_search = GridSearchCV(gbr, param_grid=grid, cv=5, n_jobs=-1)
  grid_search.fit(x_train, y_train)

  #the best model to make predictions on the test data and evaluate performance
  y_pred = grid_search.predict(x_test)

  #inverse transform the scaled predictions to get the original scale, uses a reverse of original formula
  for i in range(len(y_pred)):
    y_pred[i] = (y_pred[i]*(arr[1] - arr[0])) + arr[0]
  for i in range(len(y_test)):
    y_test[i] = (y_test[i]*(arr[1] - arr[0])) + arr[0]


  # print(mean_absolute_error(y_test, y_pred))

  return grid_search.best_params_

#gets original value for fantasy points for predictions.
def getScaleBack(df):
  #index of column
  column_index = df.columns.get_loc("PPG")

  #min value of column:
  min_value = df["PPG"].min()

  #scaling valye of column
  #scaling_factor = scaler.scale_[column_index]
  max_value = df["PPG"].max()

  #array to be used later to scale each data
  arr = [min_value, max_value]

  return arr

def test(df, model, arr):
  #make columns everything but target
  predictors = [col for col in df.columns if col != "PPG" and 'Unnamed' not in col and col != 'YearsBack']


  #make train and test sets
  x = df[predictors].values
  y = df["PPG"].values
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=40)

  #make the predictions
  predict_test = model.predict(x_test)

  #inverse transform the scaled predictions to get the original scale by reversing formula
  for i in range(len(predict_test)):
    predict_test[i] = (predict_test[i]*(arr[1] - arr[0])) + arr[0]
  for i in range(len(y_test)):
    y_test[i] = (y_test[i]*(arr[1] - arr[0])) + arr[0]

  #average error 
  mae = mean_absolute_error(y_test, predict_test)
  # print("test ", mae)

#if ppr is 0, than it is non ppr. if 1, then it is half ppr. if 2, full ppr. loops through each.
for ppr in [0,1,2]:

  dfFantasyCopy = dfFantasy.copy()

  dfFantasyCopy = correctData(dfFantasyCopy, ppr)
  
  dfFantasyCopy = makeCorrectShift(dfFantasyCopy)

  dfFantasyCopy = dfFantasyCopy.loc[dfFantasyCopy["season"] != 2012]

  dfFantasyCopy = removeUnwanted(dfFantasyCopy, "WRTE")

  dfFantasyCopy = dfFantasyCopy.reset_index(drop=True)

  #gets fantasy_points_ppr scale per each position
  scaleWRTE = getScaleBack(dfFantasyCopy)

  dfFantasyCopy[dfFantasyCopy.columns] = scaler.fit_transform(dfFantasyCopy[dfFantasyCopy.columns])

  #obtained by running the getBestParams function per each respective position
  paramWRTE = getBestParams(dfFantasyCopy, scaleWRTE)

  #makes array of model and score, then prints it
  wrteArray = machineLearning(dfFantasyCopy, scaleWRTE, paramWRTE)
  num = wrteArray[0]
  wrteModel = wrteArray[1]

  print("wrte score(ppg off on average per player): ", num)
  if ppr == 0:
      joblib.dump(wrteModel, "wrte models/wrteModelNonPPR.joblib")
  elif ppr == 1:
      joblib.dump(wrteModel, "wrte models/wrteModelHalfPPR.joblib")
  elif ppr == 2:
      joblib.dump(wrteModel, "wrte models/wrteModelPPR.joblib")

/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.         0.         ... 0.70588235 0.         0.6875    ]' has dtype incompatible with int32, please explicitly cast to a compa

receiving_air_yards: 0.07396535999326154
qb: 0.05694327892756551
wrte: 0.0345739690319771
oline: 0.03269534961098141
receiving_yards: 0.00997234175258672
dst: 0.007937596368230887
receptions: 0.0038350856098642193
rushing_fumbles_lost: 0.0
rushing_2pt_conversions: 0.0
receiving_first_downs: -0.00020705308810204493
targets: -0.0005283598149381796
rushing_first_downs: -0.0006785308147451263
special_teams_tds: -0.0007736686053106612
receiving_yards_after_catch: -0.0021134397807841734
rrtd: -0.002561134401544837
receiving_fumbles_lost: -0.004504687967197222
carries: -0.006450488397258723
receiving_tds: -0.0066511458356960425
rushing_yards: -0.008529542686143895
receiving_2pt_conversions: -0.011964110838746244
rushing_tds: -0.01459155428557986
rb: -0.018729118322915836
age: -0.06748413863887387
wrte score(ppg off on average per player):  0.11803692091176114


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.         0.         ... 0.70588235 0.         0.6875    ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.    0.    0.    ... 0.    0.    0.125]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype in

receiving_yards: 0.1593140732380729
receiving_air_yards: 0.12559640021704546
receiving_yards_after_catch: 0.02119332487105691
receptions: 0.01563368653211579
wrte: 0.010283577789193714
qb: 0.006928234692923512
receiving_fumbles_lost: 0.0037503279108837507
receiving_2pt_conversions: 0.0017145382699218859
receiving_tds: 0.0010980036078527611
age: 0.0009221800547715308
targets: 0.0006351478861580717
rushing_2pt_conversions: 0.00020503768981887217
rushing_yards: 0.00013044692431204318
carries: 4.457332430203742e-05
special_teams_tds: 0.0
rushing_fumbles_lost: -0.00010164906428614007
rb: -0.00014478296634038435
rushing_first_downs: -0.00028567752798602905
rushing_tds: -0.0007151958762790289
receiving_first_downs: -0.001995366567041843
dst: -0.003915046778170451
rrtd: -0.005037563438722943
oline: -0.0051500866191994035
wrte score(ppg off on average per player):  0.10686084265670542


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.         0.         0.         ... 0.70588235 0.         0.6875    ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.    0.    0.    ... 0.    0.    0.125]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = df[col] / df['GP']
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_12316/22791392.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype in

receiving_yards: 0.28815478228301145
receiving_air_yards: 0.03371074422414258
wrte: 0.014696891111326682
qb: 0.01350310071241071
targets: 0.010566959399701023
receptions: 0.009548217492927547
age: 0.006209716181079239
rushing_first_downs: 0.0026119765143944695
carries: 0.0017446498504074149
rb: 0.0015287422516947547
receiving_fumbles_lost: 0.001341254188191746
rrtd: 0.0012597452348444315
receiving_tds: 0.0010944178509875368
rushing_yards: 0.00048573326634651415
dst: 0.0004611746654700433
oline: 0.00021652785007861608
rushing_fumbles_lost: 0.0
rushing_2pt_conversions: 0.0
special_teams_tds: -2.9977363552354807e-05
rushing_tds: -8.146517002867837e-05
receiving_2pt_conversions: -8.796627780476407e-05
receiving_yards_after_catch: -0.0005953834234335753
receiving_first_downs: -0.0008027138292263524
wrte score(ppg off on average per player):  0.11201565130995325
